In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

خواندن داده ها

In [3]:
df=pd.read_csv("loans.csv")
df.head(10)

,client_id,loan_type,loan_amount,repaid,loan_id,loan_start,loan_end,rate
0,46109,home,13672,0,10243,2002-04-16,2003-12-20,2.15
1,46109,credit,9794,0,10984,2003-10-21,2005-07-17,1.25
2,46109,home,12734,1,10990,2006-02-01,2007-07-05,0.68
3,46109,cash,12518,1,10596,2010-12-08,2013-05-05,1.24
4,46109,credit,14049,1,11415,2010-07-07,2012-05-21,3.13
5,46109,home,6935,0,11501,2006-09-17,2008-11-26,1.94
6,46109,cash,6177,1,11141,2007-03-12,2009-04-26,9.48
7,46109,home,12656,0,11658,2006-05-26,2007-10-15,4.14
8,46109,home,11062,1,11611,2012-09-12,2014-03-14,5.48
9,46109,other,4050,1,10828,2003-12-06,2005-08-19,4.26


پاکسازی داده ها 

In [4]:
if df.duplicated().any():
    print("سطر تکراری وجود دارد")
    df.drop_duplicates(inplace=True)
else:
    print("هیچ سطر تکراری یافت نشد")

هیچ سطر تکراری یافت نشد


In [5]:
empty_rows_mask = df.isnull().all(axis=1)
if empty_rows_mask.any():
    print(f"تعداد {empty_rows_mask.sum()} سطر کاملاً خالی پیدا شد.")
    df.dropna(how='all', inplace=True)
    print("سطرهای خالی با موفقیت حذف شدند.")

else:
    print("هیچ سطر کاملاً خالی در دیتافریم وجود ندارد.")

هیچ سطر کاملاً خالی در دیتافریم وجود ندارد.


In [6]:
def clean_all_columns(df):
    for col in df.columns:
        if df[col].isnull().any():
            print(f"ستون '{col}' مقادیر خالی دارد.")

            if df.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
                print(f" -> ستون '{col}با میانه  (median) پر شد.")

            else:
                if not df[col].mode().empty:
                    df[col] = df[col].fillna(df[col].mode()[0])
                    print(f" -> ستون '{col}' با مُد (Mode) پر شد.")
                else:
                    print(f" -> اخطار: ستون '{col}' قابل پر کردن نبود (تماماً خالی است).")
        else:
            pass
    return df
df=clean_all_columns(df)

تشخیص و پردازش داده های پرت برای ستون های مهم

In [7]:
target_cols = ['loan_amount', 'rate']

for col in target_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"ستون '{col}': تعداد {len(outliers)} داده پرت شناسایی شد.")

    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
print(f"\nتعداد سطرها پس از حذف داده‌های پرت: {df.shape[0]}")

ستون 'loan_amount': تعداد 0 داده پرت شناسایی شد.
ستون 'rate': تعداد 3 داده پرت شناسایی شد.

تعداد سطرها پس از حذف داده‌های پرت: 440


تبدیل متغیرها

In [8]:
#تبدیل لگاریتمی روی مبلغ وام 

df['loan_amount_log'] = np.log1p(df['loan_amount'])
print("-> ستون 'loan_amount_log' ساخته شد (تبدیل توزیع مبلغ).")

#اطمینان از اینکه ستون های تاریخ فرمت درست دارند 
df['loan_start'] = pd.to_datetime(df['loan_start'])
df['loan_end'] = pd.to_datetime(df['loan_end'])

# محاسبه اختلاف تاریخ
df['duration_days'] = (df['loan_end'] - df['loan_start']).dt.days
print("-> ستون 'duration_days' ساخته شد (مهندسی ویژگی زمانی).")

# حذف ستون های قدیمی

df.drop(['loan_amount', 'loan_start', 'loan_end'], axis=1, inplace=True)
print("\nستون‌های نهایی پس از مهندسی ویژگی:")
print(df.columns.tolist())

-> ستون 'loan_amount_log' ساخته شد (تبدیل توزیع مبلغ).
-> ستون 'duration_days' ساخته شد (مهندسی ویژگی زمانی).

ستون‌های نهایی پس از مهندسی ویژگی:
['client_id', 'loan_type', 'repaid', 'loan_id', 'rate', 'loan_amount_log', 'duration_days']


رمگذاری متغیرها

In [19]:
# شناسایی ستون‌های متنی (بغیر از ستون‌های تاریخ که قبلاً مدیریت کردیم)
cat_cols = df.select_dtypes(include=['object']).columns

# ایجاد و اعمال رمزگذار
le =LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    
df.head(10)

,client_id,loan_type,repaid,loan_id,rate,loan_amount_log,duration_days
0,46109,2,0,10243,2.15,9.523178,613
1,46109,1,0,10984,1.25,9.189627,635
2,46109,2,1,10990,0.68,9.452109,519
3,46109,0,1,10596,1.24,9.435003,879
4,46109,1,1,11415,3.13,9.550378,684
5,46109,2,0,11501,1.94,8.844481,801
6,46109,0,1,11141,9.48,8.728750,776
7,46109,2,0,11658,4.14,9.445966,507
8,46109,2,1,11611,5.48,9.311361,548
9,46109,3,1,10828,4.26,8.306719,622


مقیاس بندی

In [13]:
#انتخاب ستون های عددی برای مقیاس بندی 
scaler = StandardScaler()

scaled_data = scaler.fit_transform(df)
df_scaled = pd.DataFrame(scaled_data, columns=df.columns)

print("--- داده‌ها مقیاس‌بندی شدند ---")
df_scaled.describe().loc[['mean', 'std']]

--- داده‌ها مقیاس‌بندی شدند ---


,client_id,loan_type,repaid,loan_id,rate,loan_amount_log,duration_days
mean,-1.877286e-16,5.652044e-17,5.248327e-17,-9.608476e-16,-1.170781e-16,5.248327e-17,-3.552714e-16
std,1.001138e+00,1.001138e+00,1.001138e+00,1.001138e+00,1.001138e+00,1.001138e+00,1.001138e+00


تقسیم داده ها به آموزش و تست

In [15]:
# تمام ستون‌ها به جز هدف
X = df_scaled.drop('repaid', axis=1)
y = df['repaid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- ابعاد داده‌های تقسیم‌بندی شده ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

--- ابعاد داده‌های تقسیم‌بندی شده ---
X_train shape: (352, 6)
X_test shape: (88, 6)
y_train shape: (352,)
y_test shape: (88,)


ذخیره سازی داده ها

In [20]:
#  ترکیب داده‌های آموزش (X_train و y_train)
train_final = pd.concat([X_train, y_train], axis=1)
train_final.to_csv('loans_train_final.csv', index=False)
print("فایل 'loans_train_final.csv' با موفقیت ذخیره شد.")

#  ترکیب داده‌های تست (X_test و y_test)
test_final = pd.concat([X_test, y_test], axis=1)
test_final.to_csv('loans_test_final.csv', index=False)
print("فایل 'loans_test_final.csv' با موفقیت ذخیره شد.")

فایل 'loans_train_final.csv' با موفقیت ذخیره شد.
فایل 'loans_test_final.csv' با موفقیت ذخیره شد.
